# 02 — Follow service time and departure events

A departure profile is not a minute-by-minute screenshot of point queries. We'll inspect the lab's actual event-based retained-label range search, then check it against an intentionally slow oracle.

Scope: one admitted service date, integer seconds, fixed walking, arrival/boarding objectives, and transit-required origin/destination pairs. See [chapter 06](../docs/06_departure_profiles_and_reverse_search.md).

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src" / "raptor.py").is_file():
    raise RuntimeError("Start this notebook from the repository root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from datetime import date
from src import compile_timetable, demo_timetable, rraptor, arrive_by, last_connection, parse_time, format_time
from src.service_time import service_instant
seconds = parse_time("24:06:00")
print("Service seconds:", seconds)
print("Seoul instant:", service_instant(date(2026, 9, 7), seconds).isoformat())
assert seconds == 86760

## Inspect the compressed intervals

The engine derives origin-ready thresholds from trip departures minus access walking and boarding slack. It processes thresholds latest to earliest and retains each boarding round's labels.

The intervals below are inclusive. A boundary at 08:00:00 means the next interval starts at 08:00:01.

In [ ]:
index = compile_timetable(demo_timetable())
start, end = parse_time("07:59:00"), parse_time("08:12:00")
profile = rraptor(index, "O", "Z", start, end, max_boardings=3, boarding_slack=60)
for segment in profile.segments:
    outcomes = [(format_time(arrival), boardings) for arrival, boardings in segment.signature]
    print(format_time(segment.ready_from), "through", format_time(segment.ready_through), outcomes or "NO_ROUTE")
assert profile.metrics.reused_labels > 0
print("Retained-label reuse:", profile.metrics.reused_labels)

## Use a slow method to check a faster one

The oracle explores a state graph and enumerates every boardable trip. Here it checks every integer second in the small window. It does not reuse the range engine's scan, marking, or walking-closure code.

That repetition belongs in a test, not in a production profile endpoint or fallback.

In [ ]:
from src.oracle import exact_profile_seconds
expected = exact_profile_seconds(index.timetable, "O", "Z", start, end, max_boardings=3, boarding_slack=60)
for ready, signature in expected.items():
    journeys = profile.at(ready)
    assert tuple((j.arrival, j.boardings) for j in journeys) == signature
    for journey in journeys:
        journey.validate(60)
print(f"All {len(expected)} integer-second queries agree with the independent oracle.")

## Reverse the question, not the physical walkway

ARRIVE_BY finds the latest feasible origin readiness before a completed-destination-arrival deadline. Backward feasibility uses indexes of existing forward edges. It doesn't invent the opposite physical direction.

In [ ]:
deadline = parse_time("08:22:00")
journey = arrive_by(index, "O", "Z", deadline, max_boardings=3, boarding_slack=60)
assert journey is not None
journey.validate(60)
print("Latest ready:", format_time(journey.ready_at))
print("Actual arrival:", format_time(journey.arrival))
assert journey.ready_at == parse_time("08:00:00")
assert arrive_by(index, "Z", "O", deadline, max_boardings=3, boarding_slack=60) is None

## Last connection isn't 23:59

This miniature morning timetable has its last feasible origin departure at 08:10. The result comes from actual supplied events, not from a special clock time. In a real service, calendars, occurrence identity, service-date spans, and realtime applicability also need to be admitted.

In [ ]:
last = last_connection(index, "O", "Z", max_boardings=3, boarding_slack=60)
assert last is not None
assert last.ready_at == parse_time("08:10:00")
assert last.arrival == parse_time("08:38:00")
print("Last ready:", format_time(last.ready_at), "Actual arrival:", format_time(last.arrival))

## Name the limitation before extending it

The lab rejects walking-only departure profiles because their arrival function can be affine (`ready + walking_duration`) rather than constant. It also doesn't implement multi-date profile merging or multicriteria representatives inside the route scan.

Question: what must change before two intervals are equivalent under a least-walking or safest-connection promise? Answer: both the intermediate frontier and the interval signature must preserve those criteria; a scalar destination filter cannot recover a path already discarded earlier.